In [ ]:
!pip install belt_nlp
!pip install datasets
!pip install accelerate -U
!pip install transformers[torch] -U

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from typing import Optional
import torch
from torch import Tensor
from transformers import BatchEncoding, PreTrainedTokenizerBase
from transformers import AutoTokenizer
from transformers import Trainer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

from belt_nlp.exceptions import InconsistentSplittingParamsException

In [ ]:
def check_split_parameters_consistency(chunk_size: int, stride: int, minimal_chunk_length: int) -> None:
    if chunk_size > 510:
        raise InconsistentSplittingParamsException("Size of each chunk cannot be bigger than 510!")
    if minimal_chunk_length > chunk_size:
        raise InconsistentSplittingParamsException("Minimal length cannot be bigger than size!")
    if stride > chunk_size:
        raise InconsistentSplittingParamsException(
            "Stride cannot be bigger than size! Chunks must overlap or be near each other!"
        )

In [ ]:
def split_tokens_into_smaller_chunks(
    tokens: BatchEncoding,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
) -> tuple[list[Tensor], list[Tensor]]:
    """Splits tokens into overlapping chunks with given size and stride."""
    input_id_chunks = split_overlapping(tokens["input_ids"][0], chunk_size, stride, minimal_chunk_length)
    mask_chunks = split_overlapping(tokens["attention_mask"][0], chunk_size, stride, minimal_chunk_length)
    return input_id_chunks, mask_chunks


In [ ]:
def split_overlapping(tensor: Tensor, chunk_size: int, stride: int, minimal_chunk_length: int) -> list[Tensor]:
    """Helper function for dividing 1-dimensional tensors into overlapping chunks."""
    check_split_parameters_consistency(chunk_size, stride, minimal_chunk_length)
    result = [tensor[i : i + chunk_size] for i in range(0, len(tensor), stride)]
    if len(result) > 1:
        # ignore chunks with less than minimal_length number of tokens
        result = [x for x in result if len(x) >= minimal_chunk_length]
    return result

In [ ]:
def add_special_tokens_at_beginning_and_end(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> None:
    """
    Adds special CLS token (token id = 101) at the beginning.
    Adds SEP token (token id = 102) at the end of each chunk.
    Adds corresponding attention masks equal to 1 (attention mask is boolean).
    """
    for i in range(len(input_id_chunks)):
        # adding CLS (token id 101) and SEP (token id 102) tokens
        input_id_chunks[i] = torch.cat([Tensor([101]), input_id_chunks[i], Tensor([102])])
        # adding attention masks  corresponding to special tokens
        mask_chunks[i] = torch.cat([Tensor([1]), mask_chunks[i], Tensor([1])])

In [ ]:
def add_padding_tokens(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> None:
    """Adds padding tokens (token id = 0) at the end to make sure that all chunks have exactly 512 tokens."""
    for i in range(len(input_id_chunks)):
        # get required padding length
        pad_len = 512 - input_id_chunks[i].shape[0]
        # check if tensor length satisfies required chunk size
        if pad_len > 0:
            # if padding length is more than 0, we must add padding
            input_id_chunks[i] = torch.cat([input_id_chunks[i], Tensor([0] * pad_len)])
            mask_chunks[i] = torch.cat([mask_chunks[i], Tensor([0] * pad_len)])

In [ ]:
def stack_tokens_from_all_chunks(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> tuple[Tensor, Tensor]:
    """Reshapes data to a form compatible with BERT model input."""
    input_ids = torch.stack(input_id_chunks)
    attention_mask = torch.stack(mask_chunks)

    return input_ids.long(), attention_mask.int()


In [ ]:
def tokenize_whole_text(text: str, tokenizer: PreTrainedTokenizerBase) -> BatchEncoding:
    """Tokenizes the entire text without truncation and without special tokens."""
    tokens = tokenizer(text, add_special_tokens=False, truncation=False, return_tensors="pt")
    return tokens

In [ ]:
def tokenize_text_with_truncation(
    text: str, tokenizer: PreTrainedTokenizerBase, maximal_text_length: int
) -> BatchEncoding:
    """Tokenizes the text with truncation to maximal_text_length and without special tokens."""
    tokens = tokenizer(text, add_special_tokens=False, max_length=maximal_text_length, truncation=True, return_tensors="pt")
    return tokens

In [ ]:
def transform_single_text(
    text: str,
    tokenizer: PreTrainedTokenizerBase,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
    maximal_text_length: Optional[int],
) -> tuple[Tensor, Tensor]:
    """Transforms (the entire) text to model input of BERT model."""
    if maximal_text_length:
        tokens = tokenize_text_with_truncation(text, tokenizer, maximal_text_length)
    else:
        tokens = tokenize_whole_text(text, tokenizer)
    input_id_chunks, mask_chunks = split_tokens_into_smaller_chunks(tokens, chunk_size, stride, minimal_chunk_length)
    add_special_tokens_at_beginning_and_end(input_id_chunks, mask_chunks)
    add_padding_tokens(input_id_chunks, mask_chunks)
    #padded_input_id_chunks, padded_mask_chunks = equalize_padding(input_id_chunks, mask_chunks)
    #input_ids, attention_mask = stack_tokens_from_all_chunks(padded_input_id_chunks, padded_mask_chunks)
    input_ids, attention_mask = stack_tokens_from_all_chunks(input_id_chunks, mask_chunks)
    return input_ids, attention_mask

In [ ]:
def transform_list_of_texts(
    texts: list[str],
    tokenizer: PreTrainedTokenizerBase,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
    maximal_text_length: Optional[int] = None,
) -> BatchEncoding:
    model_inputs = [
        transform_single_text(text, tokenizer, chunk_size, stride, minimal_chunk_length, maximal_text_length)
        for text in texts
    ]
    input_ids = [model_input[0] for model_input in model_inputs]
    attention_mask = [model_input[1] for model_input in model_inputs]
    tokens = {"input_ids": input_ids, "attention_mask": attention_mask}
    return BatchEncoding(tokens)

In [ ]:
source_file_path = '/content/drive/MyDrive/codeT5p_data_@/demo_data/train.src'
target_file_path = '/content/drive/MyDrive/codeT5p_data_@/demo_data/train.tgt'

def load_data(input_path, target_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        inputs = [line.strip() for line in file.readlines()]

    with open(target_path, 'r', encoding='utf-8') as file:
        targets = [line.strip() for line in file.readlines()]

    return inputs, targets
inputs, targets = load_data(source_file_path, target_file_path)


In [ ]:
print(type(inputs))
print(type(targets))

<class 'list'>
<class 'list'>


In [ ]:
print(len(inputs))
print(len(targets))

2
2


In [ ]:
print(inputs[0])
print(targets[0])

module entry_point ( , ap_clk, , ap_rst, , ap_start, , ap_done, , ap_idle, , ap_ready, , n, , item, , temptemp1, , temptemp1_ap_vld, , temptemp2, , temptemp2_ap_vld, , temptemp3, , temptemp3_ap_vld, , ap_return , ); , parameter ap_ST_fsm_state1 = 3'd1; , parameter ap_ST_fsm_state2 = 3'd2; , parameter ap_ST_fsm_state3 = 3'd4; , input ap_clk; , input ap_rst; , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] n; , input [31:0] item; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] temptemp2; , output temptemp2_ap_vld; , output [31:0] temptemp3; , output temptemp3_ap_vld; , output [31:0] ap_return; , reg ap_done; , reg ap_idle; , reg ap_ready; , reg temptemp1_ap_vld; , reg temptemp2_ap_vld; , reg temptemp3_ap_vld; , (* fsm_encoding = "none" *) reg [2:0] ap_CS_fsm; , wire ap_CS_fsm_state1; , wire [30:0] trunc_ln29_fu_90_p1; , reg [30:0] trunc_ln29_reg_121; , wire [0:0] icmp_ln13_fu_95_p2; , reg [0:0] icmp_ln13_reg_133; , wire [0:0] g

In [ ]:
pretrained_model_codet5p = "Salesforce/codet5p-220m"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_codet5p)

chunk_size = 510 # Chunk size for inputs
stride = 510     # Overlap size for input chunks
minimal_chunk_length = 1

# Tokenize and chunk input texts
tokenized_inputs = transform_list_of_texts(inputs, tokenizer, chunk_size, stride, minimal_chunk_length)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (6420 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
# Tokenized the targets without need of chunks .squeeze(0)
#tokenized_targets = [tokenizer(tgt, add_special_tokens=True, padding="max_length", max_length=512, truncation=True)['input_ids'] for tgt in targets]
tokenized_targets = [torch.tensor(tokenizer(tgt, add_special_tokens=True, padding="max_length", max_length=512, truncation=True)['input_ids'], dtype=torch.long) for tgt in targets]


In [ ]:
print('length of tokenized_inputs : ',len(tokenized_inputs))
print('length of tokenized_inputs[input_ids] : ',len(tokenized_inputs['input_ids']))
print('length of tokenized_inputs[input_ids] : ',len(tokenized_inputs['attention_mask']))
print()
print('length of tokenized_targets : ',len(tokenized_targets))

length of tokenized_inputs :  2
length of tokenized_inputs[input_ids] :  2
length of tokenized_inputs[input_ids] :  2

length of tokenized_targets :  2


In [ ]:
print('type of the tokenized_inputs : ',type(tokenized_inputs))
print('type of the tokenized_inputs[input_ids] : ',type(tokenized_inputs['input_ids']))
print('type of the tokenized_inputs[attention_mask] : ', type(tokenized_inputs['attention_mask']))
print()
print('type of the tokenized_targes : ', type(tokenized_targets))

type of the tokenized_inputs :  <class 'transformers.tokenization_utils_base.BatchEncoding'>
type of the tokenized_inputs[input_ids] :  <class 'list'>
type of the tokenized_inputs[attention_mask] :  <class 'list'>

type of the tokenized_targes :  <class 'list'>


In [ ]:
print('tokenized_inputs[input_ids] : ',tokenized_inputs['input_ids'])


tokenized_inputs[input_ids] :  [tensor([[  101,  2978,  1241,  ...,  6636, 14295,   102],
        [  101,    67,  4099,  ...,    67, 28677,   102],
        [  101,    67,  3437,  ...,  1285,  5127,   102],
        ...,
        [  101,  2376,   269,  ...,   273,   513,   102],
        [  101,    67,   882,  ...,    31,   269,   102],
        [  101,   960,   306,  ...,     0,     0,     0]]), tensor([[ 101, 2978, 1241,  ..., 6636,  513,  102],
        [ 101,   67, 4596,  ...,   67, 6669,  102],
        [ 101,   67, 6420,  ..., 5127,   67,  102],
        ...,
        [ 101,   67, 2556,  ...,   12,  269,  102],
        [ 101,  513,   67,  ...,  269, 2376,  102],
        [ 101,  269,  309,  ...,    0,    0,    0]])]


In [ ]:
print('tokenized_inputs[attention_mask] : ', tokenized_inputs['attention_mask'])

tokenized_inputs[attention_mask] :  [tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)]


In [ ]:
print(tokenized_inputs)

{'input_ids': [tensor([[  101,  2978,  1241,  ...,  6636, 14295,   102],
        [  101,    67,  4099,  ...,    67, 28677,   102],
        [  101,    67,  3437,  ...,  1285,  5127,   102],
        ...,
        [  101,  2376,   269,  ...,   273,   513,   102],
        [  101,    67,   882,  ...,    31,   269,   102],
        [  101,   960,   306,  ...,     0,     0,     0]]), tensor([[ 101, 2978, 1241,  ..., 6636,  513,  102],
        [ 101,   67, 4596,  ...,   67, 6669,  102],
        [ 101,   67, 6420,  ..., 5127,   67,  102],
        ...,
        [ 101,   67, 2556,  ...,   12,  269,  102],
        [ 101,  513,   67,  ...,  269, 2376,  102],
        [ 101,  269,  309,  ...,    0,    0,    0]])], 'attention_mask': [tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), tensor([[1, 1, 1,  ..., 1, 1, 1],
   

In [ ]:
# To check if any chunks are empty
# Assuming tokenized_inputs is a dictionary containing 'input_ids' and 'attention_mask'
# which are lists of lists or similar structures containing token ids.

empty_chunks_count = 0

for chunk in tokenized_inputs['attention_mask']:
    # Check if the chunk is empty or has no tokens
    if len(chunk) == 0:
        empty_chunks_count += 1

print(f"Number of empty chunks: {empty_chunks_count}")


Number of empty chunks: 0


## Dictionary

In [ ]:
from torch.utils.data import Dataset
from typing import Any, Union, Tuple
from torch.utils.data import Dataset, RandomSampler, DataLoader

class Seq2SeqDataset(Dataset):
    def __init__(self, tokens: BatchEncoding, labels: Optional[list] = None):
        self.input_ids = tokens["input_ids"]
        self.attention_mask = tokens["attention_mask"]
        self.labels = labels

    def __len__(self) -> int:
        return len(self.input_ids)

    # def __getitem__(self, idx: int) -> Union[tuple[Tensor, Tensor, Any], tuple[Tensor, Tensor]]:
    #     if self.labels:
    #         return self.input_ids[idx], self.attention_mask[idx], self.labels[idx]
    #     return self.input_ids[idx], self.attention_mask[idx]
    def __getitem__(self, idx: int) -> dict:
        if self.labels:
            return {
                'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'labels': self.labels[idx]
            }


In [ ]:
# Initialize your dataset
dataset_dict = Seq2SeqDataset(tokenized_inputs, tokenized_targets)

In [ ]:
loader_dict = DataLoader(dataset_dict, sampler=RandomSampler(dataset_dict), batch_size=1)

In [ ]:
try:
    for batch in dataset_dict:
        print(batch)
        break
except RuntimeError as e:
    print(e)

{'input_ids': tensor([[  101,  2978,  1241,  ...,  6636, 14295,   102],
        [  101,    67,  4099,  ...,    67, 28677,   102],
        [  101,    67,  3437,  ...,  1285,  5127,   102],
        ...,
        [  101,  2376,   269,  ...,   273,   513,   102],
        [  101,    67,   882,  ...,    31,   269,   102],
        [  101,   960,   306,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), 'labels': tensor([    1,     7,  6702,   411,  5084,  1594,    18,    76,    34,   632,
          225,   632,   918,   855,  1076,   261,   509,   279,   306,   225,
          308,   225,   269,   225,   509,   963,   262,   282,   288,   632,
          509,   277,   274,   632,   364,   225,   261,   277,   273,   374,
          274,   225,   277,   411,   963,   

In [ ]:
from transformers import AutoModelForSeq2SeqLM

pretrained_model_codet5p = "Salesforce/codet5p-220m"
model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_codet5p)


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=1,   # batch size for training
    per_device_eval_batch_size=1,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    do_eval=True,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict,
    eval_dataset=dataset_dict,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()


ValueError: too many values to unpack (expected 2)

## list

In [ ]:
from torch.utils.data import Dataset
from typing import Any, Union, Tuple
from torch.utils.data import Dataset, RandomSampler, DataLoader

class Seq2SeqDataset_(Dataset):
    def __init__(self, tokens: BatchEncoding, labels: Optional[list] = None):
        self.input_ids = tokens["input_ids"]
        self.attention_mask = tokens["attention_mask"]
        self.labels = labels

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> Union[tuple[Tensor, Tensor, Any], tuple[Tensor, Tensor]]:
        if self.labels:
            return self.input_ids[idx], self.attention_mask[idx], self.labels[idx]
        return self.input_ids[idx], self.attention_mask[idx]


In [ ]:
dataset_list = Seq2SeqDataset_(tokenized_inputs, tokenized_targets)


In [ ]:
try:
    for batch in dataset_list:
        print(batch)
        break
except RuntimeError as e:
    print(e)

(tensor([[  101,  2978,  1241,  ...,  6636, 14295,   102],
        [  101,    67,  4099,  ...,    67, 28677,   102],
        [  101,    67,  3437,  ...,  1285,  5127,   102],
        ...,
        [  101,  2376,   269,  ...,   273,   513,   102],
        [  101,    67,   882,  ...,    31,   269,   102],
        [  101,   960,   306,  ...,     0,     0,     0]]), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), tensor([    1,     7,  6702,   411,  5084,  1594,    18,    76,    34,   632,
          225,   632,   918,   855,  1076,   261,   509,   279,   306,   225,
          308,   225,   269,   225,   509,   963,   262,   282,   288,   632,
          509,   277,   274,   632,   364,   225,   261,   277,   273,   374,
          274,   225,   277,   411,   963,   274,   225,   277,  9904,   262,
        

In [ ]:
from torch import Tensor

def collate_fn_pooled_tokens(data):
    #print(len(data))
    input_ids = [data[i][0] for i in range(len(data))]

    attention_mask = [data[i][1] for i in range(len(data))]
    if len(data[0]) == 2:
        collated = [input_ids, attention_mask]
    else:
        labels = [data[i][2] for i in range(len(data))]
        collated = [input_ids, attention_mask, labels]
    return collated

In [ ]:
loader_list = DataLoader(dataset_list, sampler=RandomSampler(dataset_list), batch_size=1, collate_fn=collate_fn_pooled_tokens)

In [ ]:
try:
    for batch in loader_list:
        print(batch)
        break
except RuntimeError as e:
    print(e)

[[tensor([[ 101, 2978, 1241,  ..., 6636,  513,  102],
        [ 101,   67, 4596,  ...,   67, 6669,  102],
        [ 101,   67, 6420,  ..., 5127,   67,  102],
        ...,
        [ 101,   67, 2556,  ...,   12,  269,  102],
        [ 101,  513,   67,  ...,  269, 2376,  102],
        [ 101,  269,  309,  ...,    0,    0,    0]])], [tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)], [tensor([    1,     7,  6702,   411,  5084,  1594,    18,    76,    34,   632,
          225,   632,   509,  1241,    67,  1153,   261,   509,   261,   380,
         1467,   262,   225,   269,   225,   509,   261,   380,  5399,   262,
          225,   269,   225,   509,   261,   380,   874,   337, 30752,    21,
          262,   225,   269,   225,   509,   261,   380,   874,   337, 30752,
           22,   262,   225,   262,   2

In [ ]:
from transformers import AutoModelForSeq2SeqLM

pretrained_model_codet5p = "Salesforce/codet5p-220m"
model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_codet5p)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=1,   # batch size for training
    per_device_eval_batch_size=1,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    do_eval=True,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_list,
    eval_dataset=dataset_list,
    tokenizer=tokenizer
)

trainer.train()


AttributeError: 'list' object has no attribute 'keys'

In [ ]:
# batch_size = 16
# model_name = model_checkpoint.split("/")[-1]
# output_dir = f"{model_name}-finetuned-{dataset_name}"

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, AdamW
from torch.utils.data import DataLoader
import torch

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
model_name = "Salesforce/codeT5-base" # or another variant of CodeT5

model = T5ForConditionalGeneration.from_pretrained(model_name)


optimizer = AdamW(model.parameters(), lr=5e-5)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
model.train()
num_epochs = 3  # Define the number of epochs

for epoch in range(num_epochs):
    for batch in dataset_dict:
        print(batch)
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"Epoch {epoch}, Loss: {loss.item()}")

        break  # Remove this in actual training to train on all batches

{'input_ids': tensor([[  101,  2978,  1241,  ...,  6636, 14295,   102],
        [  101,    67,  4099,  ...,    67, 28677,   102],
        [  101,    67,  3437,  ...,  1285,  5127,   102],
        ...,
        [  101,  2376,   269,  ...,   273,   513,   102],
        [  101,    67,   882,  ...,    31,   269,   102],
        [  101,   960,   306,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), 'labels': tensor([    1,     7,  6702,   411,  5084,  1594,    18,    76,    34,   632,
          225,   632,   918,   855,  1076,   261,   509,   279,   306,   225,
          308,   225,   269,   225,   509,   963,   262,   282,   288,   632,
          509,   277,   274,   632,   364,   225,   261,   277,   273,   374,
          274,   225,   277,   411,   963,   